# Tune reduced and full model

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.utils import col_config_sensitivity
from src.utils import ALPHA_LIST
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shutil import rmtree
import joblib
import torch

In [ ]:
SEG_TYPES = [
    "turns",
    "pt_noshort",
]
MODEL_NAMES = [
    "Llama-3.3-8B-Instruct",
    "Llama-3.3-70B-Instruct",
]
IMPACT_TYPES = ["self", "parent"]

In [ ]:
DF_DIR = BASE_PATH / "data" / "cleaned"
REDUCED_SAVE_DIR = BASE_PATH / "initial_results" / "reduced_preds"
FULL_SAVE_DIR = BASE_PATH / "initial_results" / "block_preds"

## Tune with prob vectors + questionaire

In [ ]:
N_THREADS = 4
N_JOBS = 2
N_REPEATS = 10
N_SPLITS = 4

In [ ]:
import json


def create_cmd_str(
    impact_type,
    alpha_val,
    df_path,
    base_save_dir,
    add_outcome_probs_flag,
    seed,
    n_repeats,
    split_tol,
    n_splits,
    n_jobs,
    max_iter=5000,
):
    """
    Create command for lr_preds

    TODO: Check if good file already exists for passed in configuration
    """
    cmd_str = f"export PYTHONPATH={BASE_PATH}; \
			export OMP_NUM_THREADS={N_THREADS}; \
			uv run python -m src.lr_preds \
			--impact_type {impact_type} \
			--alpha_val {alpha_val}\
			--df_path {df_path}\
			--base_save_dir {base_save_dir}\
            --add_outcome_probs_flag {add_outcome_probs_flag}\
			--seed {seed}\
			--n_repeats {n_repeats}\
			--split_tol {split_tol}\
			--n_splits {n_splits}\
			--n_jobs {n_jobs}\
			--max_iter {max_iter}"
    return " ".join(cmd_str.split())

### Reduced model

In [ ]:
reduced_cmd_list = []
for impact_type in IMPACT_TYPES:
    for alpha_val in ALPHA_LIST:
        df_path = DF_DIR / "processed" / f"{impact_type}.parquet"
        save_dir = REDUCED_SAVE_DIR / impact_type / f"alpha_{alpha_val}"
        cmd_str = create_cmd_str(
            impact_type=impact_type,
            alpha_val=alpha_val,
            df_path=df_path,
            base_save_dir=save_dir,
            add_outcome_probs_flag="False",
            seed=SEED,
            n_repeats=N_REPEATS,
            split_tol=0.1,
            n_splits=N_SPLITS,
            n_jobs=N_JOBS,
        )
        reduced_cmd_list.append(cmd_str)
print(f"Num commands: {len(reduced_cmd_list)}")

In [ ]:
import subprocess
from shutil import rmtree

if REDUCED_SAVE_DIR.exists():
    rmtree(REDUCED_SAVE_DIR)
REDUCED_SAVE_DIR.mkdir(exist_ok=False, parents=True)
for iteration, cmd in enumerate(reduced_cmd_list):
    print(f"{iteration +1}/{len(reduced_cmd_list)}...")
    subprocess.run(cmd, shell=True, check=True)

### Full model

In [ ]:
full_cmd_list = []
for impact_type in IMPACT_TYPES:
    for seg_type in SEG_TYPES:
        for model_name in MODEL_NAMES:
            df_path = (
                DF_DIR
                / "block"
                / "w_probs"
                / seg_type
                / model_name
                / f"{impact_type}.parquet"
            )
            for alpha_val in ALPHA_LIST:
                save_dir = (
                    FULL_SAVE_DIR
                    / impact_type
                    / seg_type
                    / model_name
                    / f"alpha_{alpha_val}"
                )
                cmd_str = create_cmd_str(
                    impact_type=impact_type,
                    alpha_val=alpha_val,
                    df_path=df_path,
                    base_save_dir=save_dir,
                    add_outcome_probs_flag="True",
                    seed=SEED,
                    n_repeats=N_REPEATS,
                    split_tol=0.1,
                    n_splits=N_SPLITS,
                    n_jobs=N_JOBS,
                    max_iter=20000,  # need more iters to converge bc of higher dimension w/ embeddings
                )
                full_cmd_list.append(cmd_str)
print(f"Num commands: {len(full_cmd_list)}")

In [ ]:
import subprocess
from shutil import rmtree

if FULL_SAVE_DIR.exists():
    rmtree(FULL_SAVE_DIR)
FULL_SAVE_DIR.mkdir(exist_ok=False, parents=True)
for iteration, cmd in enumerate(full_cmd_list):
    print(f"{iteration +1}/{len(full_cmd_list)}...")
    subprocess.run(cmd, shell=True, check=True)

## Get Best Params

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

In [ ]:
def get_best_params_score(import_dir, show_progress=False, optimize="mse"):
    if optimize not in ["mse", "r2"]:
        raise ValueError(f"optimize param must be one of ['mse', 'r2'], got {optimize}")
    best_params_score_dict = {
        "avg_mse": 9999999,
        "stdev_mse": 9999999,
        "avg_r2": 0,
        "stdev_r2": 9999999,
        "l1": "None",
        "alpha": "None",
    }
    total_files = sum(
        1
        for alpha_dir in import_dir.iterdir()
        for l1_dir in alpha_dir.iterdir()
        for repeat_dir in l1_dir.iterdir()
        for _ in repeat_dir.iterdir()
    )
    with tqdm(
        total=total_files,
        desc="Scoring params",
        leave=False,  # disappears when done
        dynamic_ncols=True,
        disable=not show_progress,
    ) as pbar:
        for alpha_dir in import_dir.iterdir():
            alpha_val = alpha_dir.name.split("_")[1]
            for l1_dir in alpha_dir.iterdir():
                l1_val = l1_dir.name.split("_")[1]
                mse_list = []
                r2_list = []
                for repeat_dir in l1_dir.iterdir():
                    for split_file in repeat_dir.iterdir():
                        df_import = pd.read_parquet(split_file)
                        mse = mean_squared_error(
                            y_true=df_import["true"], y_pred=df_import["pred"]
                        )
                        mse_list.append(mse)
                        r2 = r2_score(
                            y_true=df_import["true"], y_pred=df_import["pred"]
                        )
                        r2_list.append(r2)
                        pbar.update(1)
                # get avg mse across folds/repeats for l1-alpha combo
                avg_mse = np.mean(mse_list)
                avg_r2 = np.mean(r2_list)
                mse_condition = (optimize == "mse") and (
                    avg_mse < best_params_score_dict["avg_mse"]
                )
                r2_condition = (optimize == "r2") and (
                    avg_r2 > best_params_score_dict["best_r2"]
                )

                if mse_condition or r2_condition:
                    stdev_mse = np.std(mse_list)
                    stdev_r2 = np.std(r2_list)
                    best_params_score_dict = {
                        "avg_mse": round(avg_mse, 3),
                        "stdev_mse": round(stdev_mse, 3),
                        "avg_r2": round(avg_r2, 4),
                        "stdev_r2": round(stdev_r2, 4),
                        "l1": l1_val,
                        "alpha": alpha_val,
                    }
    return best_params_score_dict

### Reduced

In [ ]:
df_rows = []
for impact_dir in REDUCED_SAVE_DIR.iterdir():
    impact_name = impact_dir.name
    print(f"{impact_name}...")
    cur_config = {"impact_type": impact_name, "full_reduced": "reduced"}
    best_params_score_dict = get_best_params_score(
        impact_dir, show_progress=True, optimize="mse"
    )
    new_row = {
        **cur_config,
        **best_params_score_dict,
    }
    df_rows.append(new_row)
reduced_result_df = pd.DataFrame(df_rows)
display(reduced_result_df)

### Full

In [ ]:
df_rows = []
for impact_dir in FULL_SAVE_DIR.iterdir():
    impact_name = impact_dir.name
    print(f"{impact_name}...")
    for seg_dir in impact_dir.iterdir():
        seg_type = seg_dir.name
        print(f"\t{seg_type}...")
        for model_dir in seg_dir.iterdir():
            model_name = model_dir.name
            print(f"\t\t{model_name}...")
            cur_config = {
                "seg_type": seg_type,
                "model_name": model_name,
                "impact_type": impact_name,
                "full_reduced": "full_block",
            }
            best_params_score_dict = get_best_params_score(
                model_dir, show_progress=True, optimize="mse"
            )
            new_row = {
                **cur_config,
                **best_params_score_dict,
            }
            df_rows.append(new_row)
full_result_df = pd.DataFrame(df_rows)
display(full_result_df)

Export tables

In [ ]:
export_dir = BASE_PATH / "initial_results"
## Reduced
reduced_path = export_dir / "optimal_params_reduced.tsv"
if reduced_path.exists():
    reduced_path.unlink()
reduced_result_df.to_csv(reduced_path, sep="\t")
## Full
full_path = export_dir / "optimal_params_block.tsv"
if full_path.exists():
    full_path.unlink()
full_result_df.to_csv(full_path, sep="\t")